In [15]:
#Source: https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/evaluate_rag_gen_ai_evaluation_service_sdk.ipynb

import vertexai
import inspect
import logging
import warnings

# General
from IPython.display import HTML, Markdown, display
import pandas as pd
import plotly.graph_objects as go

# Main
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples, PointwiseMetric

In [16]:
# Configuración del cliente de Vertex AI
PROJECT_ID = "dataton-2024-team-01-cofares"
LOCATION = "us-central1"
EXPERIMENT = "rag-eval-01"

vertexai.init(project=PROJECT_ID, location=LOCATION)

logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

In [17]:
# ----------------------Helper functions----------------------

def print_doc(function):
    print(f"{function.__name__}:\n{inspect.getdoc(function)}\n")


def display_eval_report(eval_result, metrics=None):
    """Display the evaluation results."""

    title, summary_metrics, report_df = eval_result
    metrics_df = pd.DataFrame.from_dict(summary_metrics, orient="index").T
    if metrics:
        metrics_df = metrics_df.filter(
            [
                metric
                for metric in metrics_df.columns
                if any(selected_metric in metric for selected_metric in metrics)
            ]
        )
        report_df = report_df.filter(
            [
                metric
                for metric in report_df.columns
                if any(selected_metric in metric for selected_metric in metrics)
            ]
        )

    # Display the title with Markdown for emphasis
    display(Markdown(f"## {title}"))

    # Display the metrics DataFrame
    display(Markdown("### Summary Metrics"))
    display(metrics_df)

    # Display the detailed report DataFrame
    display(Markdown("### Report Metrics"))
    display(report_df)


def display_explanations(df, metrics=None, n=1):
    style = "white-space: pre-wrap; width: 800px; overflow-x: auto;"
    df = df.sample(n=n)
    if metrics:
        df = df.filter(
            ["instruction", "context", "reference", "completed_prompt", "response"]
            + [
                metric
                for metric in df.columns
                if any(selected_metric in metric for selected_metric in metrics)
            ]
        )

    for index, row in df.iterrows():
        for col in df.columns:
            display(HTML(f"{col}: {row[col]}"))
        display(HTML(""))


def plot_radar_plot(eval_results, max_score=5, metrics=None):
    fig = go.Figure()

    for eval_result in eval_results:
        title, summary_metrics, report_df = eval_result

        if metrics:
            summary_metrics = {
                k: summary_metrics[k]
                for k, v in summary_metrics.items()
                if any(selected_metric in k for selected_metric in metrics)
            }

        fig.add_trace(
            go.Scatterpolar(
                r=list(summary_metrics.values()),
                theta=list(summary_metrics.keys()),
                fill="toself",
                name=title,
            )
        )

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, max_score])), showlegend=True
    )

    fig.show()


def plot_bar_plot(eval_results, metrics=None):
    fig = go.Figure()
    data = []

    for eval_result in eval_results:
        title, summary_metrics, _ = eval_result
        if metrics:
            summary_metrics = {
                k: summary_metrics[k]
                for k, v in summary_metrics.items()
                if any(selected_metric in k for selected_metric in metrics)
            }

        data.append(
            go.Bar(
                x=list(summary_metrics.keys()),
                y=list(summary_metrics.values()),
                name=title,
            )
        )

    fig = go.Figure(data=data)

    # Change the bar mode
    fig.update_layout(barmode="group")
    fig.show()

In [18]:
# ----------------------Dataset----------------------

"""To evaluate the RAG generated answers,
the evaluation dataset is required to contain the following fields:

Prompt: The user supplied prompt consisting of the User Question and the RAG Retrieved Context
Response: The RAG Generated Answer

Your dataset must include a minimum of one evaluation example.
We recommend around 100 examples to ensure high-quality aggregated metrics
and statistically significant results."""

#El siguiente template a sido generado con Gemini a modo de ejemplo de implementación

import pandas as pd

# Ejemplos de preguntas de los usuarios
questions = [
    "Busco una crema para las estrías",
    "Necesito un suplemento de vitamina D para personas mayores",
    "¿Tienes algún producto para la caída del cabello?",
    "Quiero una crema hidratante para piel sensible",
    "¿Hay algún spray nasal para alergias?"
]

# Contexto recuperado por el sistema RAG (simulación de descripciones de productos relevantes)
retrieved_contexts = [
    "CREMA ACEITE ROSA MOSQU 50ML: Contiene aceite de rosa mosqueta, ideal para mejorar la apariencia de estrías. Pack Uresim Serum Ác.Hialurónico: hidrata y mejora la elasticidad de la piel.",
    "Vitamina D3 1000 IU: formulado especialmente para personas mayores, ayuda a mejorar la salud ósea. CalciD3: suplemento combinado de calcio y vitamina D para fortalecer huesos.",
    "Shampoo anti-caída con biotina: fortalece el cabello y reduce la caída. Tónico capilar de romero: revitaliza el cuero cabelludo y favorece el crecimiento.",
    "Crema hidratante Avène para piel sensible: reduce rojeces e hidrata profundamente. Eucerin UltraSENSITIVE: fórmula calmante para piel reactiva.",
    "Spray nasal antialérgico con azelastina: alivia los síntomas de alergia. Rhinomer Fuerza Suave: spray de agua de mar para limpiar fosas nasales y reducir congestión."
]

# Respuestas generadas por el sistema RAG
generated_answers = [
    "Aquí tienes opciones: CREMA ACEITE ROSA MOSQU 50ML, ideal para estrías, y Pack Uresim Serum Ác.Hialurónico, que mejora la elasticidad de la piel.",
    "Te recomiendo Vitamina D3 1000 IU y CalciD3, ambos beneficiosos para la salud ósea en personas mayores.",
    "Para la caída del cabello, prueba el shampoo con biotina y el tónico de romero para fortalecer y revitalizar.",
    "Para piel sensible, puedes utilizar Avène o Eucerin UltraSENSITIVE, ambas hidratantes calmantes.",
    "Disponemos de spray nasal con azelastina para alergias y Rhinomer Fuerza Suave para descongestionar."
]

# Creación del DataFrame para el dataset de evaluación
eval_dataset = pd.DataFrame(
    {
        "prompt": [
            "Consulta: " + question + " Contexto: " + context
            for question, context in zip(questions, retrieved_contexts)
        ],
        "response": generated_answers,
    }
)

# Mostrar el dataset de evaluación
eval_dataset

,prompt,response
0,Consulta: Busco una crema para las estrías Con...,Aquí tienes opciones: CREMA ACEITE ROSA MOSQU ...
1,Consulta: Necesito un suplemento de vitamina D...,"Te recomiendo Vitamina D3 1000 IU y CalciD3, a..."
2,Consulta: ¿Tienes algún producto para la caída...,"Para la caída del cabello, prueba el shampoo c..."
3,Consulta: Quiero una crema hidratante para pie...,"Para piel sensible, puedes utilizar Avène o Eu..."
4,Consulta: ¿Hay algún spray nasal para alergias...,Disponemos de spray nasal con azelastina para ...


In [19]:
# ----------------------Metricas----------------------

"""Select and create metrics
You can run evaluation for just one metric, or a combination of metrics.
For this example, we select a few RAG-related predefined metrics, and create a few of our own custom metrics."""

# Explore predefined metrics: https://cloud.google.com/vertex-ai/generative-ai/docs/models/metrics-templates
# See all the available metric examples

MetricPromptTemplateExamples.list_example_metric_names()

['coherence',
 'fluency',
 'safety',
 'groundedness',
 'instruction_following',
 'verbosity',
 'text_quality',
 'summarization_quality',
 'question_answering_quality',
 'multi_turn_chat_quality',
 'multi_turn_safety',
 'pairwise_coherence',
 'pairwise_fluency',
 'pairwise_safety',
 'pairwise_groundedness',
 'pairwise_instruction_following',
 'pairwise_verbosity',
 'pairwise_text_quality',
 'pairwise_summarization_quality',
 'pairwise_question_answering_quality',
 'pairwise_multi_turn_chat_quality',
 'pairwise_multi_turn_safety']

In [20]:
# See the prompt example for one of the pointwise metrics
print(MetricPromptTemplateExamples.get_prompt_template("question_answering_quality"))


# Instruction
You are an expert evaluator. Your task is to evaluate the quality of the responses generated by AI models.
We will provide you with the user input and an AI-generated response.
You should first read the user input carefully for analyzing the task, and then evaluate the quality of the responses based on the Criteria provided in the Evaluation section below.
You will assign the response a rating following the Rating Rubric and Evaluation Steps. Give step-by-step explanations for your rating, and only choose ratings from the Rating Rubric.


# Evaluation
## Metric Definition
You will be assessing question answering quality, which measures the overall quality of the answer to the question in user input. The instruction for performing a question-answering task is provided in the user prompt.

## Criteria
Instruction following: The response demonstrates a clear understanding of the question answering task instructions, satisfying all of the instruction's requirements.
Grounded

In [21]:
#Run evaluation with your dataset

rag_eval = EvalTask(
    dataset=eval_dataset,
    metrics=[
        "question_answering_quality",
        "fluency",
        "groundedness",
        "safety",
        "instruction_following",
    ],
    experiment=EXPERIMENT,
)

In [22]:
result_rag_a = rag_eval.evaluate()

Associating projects/793914295237/locations/us-central1/metadataStores/default/contexts/rag-eval-01-d8d18a8e-dff3-4678-80c4-758bb22695f2 to Experiment: rag-eval-01


INFO:google.cloud.aiplatform.metadata.experiment_resources:Associating projects/793914295237/locations/us-central1/metadataStores/default/contexts/rag-eval-01-d8d18a8e-dff3-4678-80c4-758bb22695f2 to Experiment: rag-eval-01


Computing metrics with a total of 20 Vertex Gen AI Evaluation Service API requests.


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 20 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 20/20 [00:23<00:00,  1.16s/it]

All 20 metric requests are successfully computed.



INFO:vertexai.evaluation._evaluation:All 20 metric requests are successfully computed.


Evaluation Took:23.1440806669998 seconds


INFO:vertexai.evaluation._evaluation:Evaluation Took:23.1440806669998 seconds


In [23]:
#------------- Display evaluation results -------------
"""View summary results
If you want to have an overall view of all the metrics from individual model's evaluation
result in one table, you can use the display_eval_report() helper function."""

display_eval_report(
    (
        "Model A Eval Result",
        result_rag_a.summary_metrics,
        result_rag_a.metrics_table,
    )
)

## Model A Eval Result

### Summary Metrics

,row_count,question_answering_quality/mean,question_answering_quality/std,groundedness/mean,groundedness/std,safety/mean,safety/std,instruction_following/mean,instruction_following/std
0,5.0,4.6,0.547723,1.0,0.0,1.0,0.0,4.8,0.447214


### Report Metrics

,prompt,response,question_answering_quality/explanation,question_answering_quality/score,groundedness/explanation,groundedness/score,safety/explanation,safety/score,instruction_following/explanation,instruction_following/score
0,Consulta: Busco una crema para las estrías Con...,Aquí tienes opciones: CREMA ACEITE ROSA MOSQU ...,Instruction Following: The response follows th...,5.0,The response mentions both products from the p...,1.0,"The response is safe, as it only suggests two ...",1.0,The prompt asks for a cream for stretch marks....,5.0
1,Consulta: Necesito un suplemento de vitamina D...,"Te recomiendo Vitamina D3 1000 IU y CalciD3, a...",The response follows the instructions and is g...,5.0,"The response recommends ""Vitamina D3 1000 IU"" ...",1.0,The AI models' response recommends two vitamin...,1.0,The prompt asks for a vitamin D supplement for...,4.0
2,Consulta: ¿Tienes algún producto para la caída...,"Para la caída del cabello, prueba el shampoo c...",Instruction Following: The response addresses ...,4.0,The AI-generated response exclusively uses inf...,1.0,The AI response recommends shampoo with biotin...,1.0,STEP 1: Assess instruction understanding: The ...,5.0
3,Consulta: Quiero una crema hidratante para pie...,"Para piel sensible, puedes utilizar Avène o Eu...",Instruction Following: The response follows th...,4.0,The response synthesizes information from the ...,1.0,The response recommends two moisturizer brands...,1.0,STEP 1: Assess instruction understanding: The ...,5.0
4,Consulta: ¿Hay algún spray nasal para alergias...,Disponemos de spray nasal con azelastina para ...,Instruction Following: The response follows th...,5.0,"The response mentions ""spray nasal con azelast...",1.0,The response is safe. It provides factual info...,1.0,STEP 1: Assess instruction understanding: The ...,5.0


In [ ]:
#------------- Visualize evaluation results -------------
eval_results = []
eval_results.append(
    ("Model A", result_rag_a.summary_metrics, result_rag_a.metrics_table)
)    

plot_radar_plot(
    eval_results,
    metrics=[
        f"{metric}/mean"
        # Edit your list of metrics here if you used other metrics in evaluation.
        for metric in [
            "question_answering_quality",
            "fluency",
            "groundedness",
            "safety",
            "instruction_following",
        ]
    ],
)
     

plot_bar_plot(
    eval_results,
    metrics=[
        f"{metric}/mean"
        for metric in [
            "question_answering_quality",
            "fluency",
            "groundedness",
            "safety",
            "instruction_following",
        ]
    ],
)